# 第8回：クラスを予測する—分類

**今日の問い：正解率だけで十分なのはどんなときか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 混同行列の4区分を利用場面に結びつける
- precision・recall・F1を使い分ける
- 確率と閾値を分けて考える

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- precision：陽性予測のうち正しかった割合
- recall：実際の陽性を見つけた割合
- F1：precisionとrecallの調和平均
- 閾値：確率をクラスへ変換する境界

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for candidate in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP"]:
    if candidate in available_fonts:
        plt.rcParams["font.family"] = candidate
        break

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## TRY：閾値0.5で混同行列を読む


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


## TRY：判定閾値を変える


In [ ]:
rows=[]
for threshold in [0.3, 0.5, 0.7]:
    pred=(probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


## 話し合い

活性候補を見逃したくない探索段階ならrecall、追試コストが非常に高い絞り込み段階ならprecisionを重く見る、といった使い分けが考えられます。正解は利用場面で変わります。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- 同じ確率でも閾値によりクラスが変わる
- 不均衡データではaccuracyが高くても役立たないことがある
- 閾値はモデル学習後にも業務要件から調整できる


In [ ]:
from sklearn.metrics import precision_recall_curve
precision_curve, recall_curve, thresholds = precision_recall_curve(y_valid, probability)
curve = pd.DataFrame({"閾値": thresholds, "precision": precision_curve[:-1], "recall": recall_curve[:-1]})
curve["F1"] = 2 * curve["precision"] * curve["recall"] / (curve["precision"] + curve["recall"])
display(curve.iloc[::max(1, len(curve)//10)].round(3))
best_row = curve.loc[curve["F1"].idxmax()]
print("この検証データ上でF1最大の閾値（最終性能ではない）:", round(best_row["閾値"], 3))


## よくある誤り

- 常に閾値0.5を使う
- 偽陽性と偽陰性のコストを同じとみなす
- 検証データで閾値を細かく最適化しすぎる

## SELF-STUDY（任意・30〜60分）

- 0.1刻みの閾値表を作り利用目的に合う点を選ぶ
- 探索段階と確証段階で重視する指標を比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 偽陽性・偽陰性はそれぞれ何か
2. accuracyが危険な例は何か
3. 閾値を下げると一般にrecallはどうなるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
